# 07 — Contrôle qualité, audit de fuite, test de fumée

```text
Contrôle qualité + audit de fuite + test de fumée du retrieval sur la release complète.
Sortie : artifacts/qc/qc_report.json (+ qc_report.md). Code retour 1 si un contrôle bloquant échoue.
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [15]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [16]:
import sys
from collections import Counter
import numpy as np
import pandas as pd
from common import clean_caption, get_logger, load_config, paths, read_json, read_jsonl, sha256_ids, write_json
from rag_data import Release
log = get_logger("qc")
RESULTS = []

### `check()`

In [17]:
def check(name, ok, detail="", blocking=True):
    RESULTS.append({"check": name, "status": "PASS" if ok else ("FAIL" if blocking else "WARN"), "detail": detail})
    (log.info if ok else log.error if blocking else log.warning)("%-46s %s %s", name, "PASS" if ok else "FAIL" if blocking else "WARN", detail)

### `cider_scores()`

In [18]:
def cider_scores(res: dict, gts: dict):
    from pycocoevalcap.cider.cider import Cider
    s, per = Cider().compute_score({k: [clean_caption(v) for v in gts[k]] for k in gts}, {k: [clean_caption(v)] for k, v in res.items()})
    return float(s), dict(zip(sorted(gts), per))   # compute_score trie les ids

### `main()`

In [19]:
def main():
    cfg = load_config(); p = paths(cfg); R = Release(p["art"], cfg["primary_encoder"])
    sz, ncap = cfg["sizes"], cfg["captions_per_image"]
    meta_splits = read_json(p["splits"] / "splits_meta.json")
    imgs = pd.read_csv(p["tables"] / "images.csv")
    ids = {s: set(r["image_id"] for r in R.splits[s]) for s in R.splits}

    # ---- 1. effectifs & intégrité des IDs
    for s in ids:
        check(f"taille split {s}", len(ids[s]) == sz[s], f"{len(ids[s])} vs {sz[s]}")
        check(f"hash des IDs {s} = splits_meta", sha256_ids(ids[s]) == meta_splits["id_hash_sha256"][s])
    check("splits deux à deux disjoints", not (ids["kb"] & ids["val"]) and not (ids["kb"] & ids["test"]) and not (ids["val"] & ids["test"]))
    check("images invalides (corrompues/absentes)", (imgs.status != "ok").sum() == 0, f"{(imgs.status != 'ok').sum()}")
    check("IDs uniques dans images.csv", not imgs.duplicated(["image_id"]).any())
    ok = imgs[imgs.status == "ok"]
    check("doublons exacts (sha256) inter/intra splits", not ok.duplicated("sha256_raw").any(), f"{ok.duplicated('sha256_raw', keep=False).sum()} images")
    na_cols = ["brightness", "contrast", "colorfulness", "dhash"]
    check("pas de NaN dans les stats pixel", not ok[na_cols].isna().any().any())

    # ---- 2. captions
    kbc = R.kb_captions(); ev = {s: read_jsonl(p["tables"] / f"eval_refs_{s}.jsonl") for s in ("val", "test")}
    cnt = Counter(c["image_id"] for c in kbc)
    check("KB : 5 captions / image", set(cnt) == ids["kb"] and set(cnt.values()) == {ncap}, f"min={min(cnt.values())} max={max(cnt.values())}")
    for s in ("val", "test"):
        c = Counter(r["image_id"] for r in ev[s])
        check(f"{s} : 5 références / image", set(c) == ids[s] and set(c.values()) == {ncap})
    kb_cap_ids = {c["caption_id"] for c in kbc}
    check("caption_id uniques (KB)", len(kb_cap_ids) == len(kbc))
    check("ANTI-FUITE : aucune image val/test dans kb_captions", not ({c["image_id"] for c in kbc} & (ids["val"] | ids["test"])))
    check("ANTI-FUITE : aucun caption_id d'éval dans la KB", not (kb_cap_ids & {r["caption_id"] for s in ev for r in ev[s]}))
    check("captions brutes inchangées (raw != clean autorisé, raw non vide)", all(c["caption_raw"].strip() for c in kbc))
    kbset = {clean_caption(c["caption_clean"]) for c in kbc}
    for s in ("val", "test"):
        ov = sum(clean_caption(r["caption_raw"]) in kbset for r in ev[s]) / len(ev[s])
        check(f"info : refs {s} identiques à une caption KB (texte)", True, f"{100 * ov:.1f}% — normal pour des captions génériques; pas une fuite d'image")

    # ---- 3. métadonnées
    M = R.metadata()
    check("métadonnées pour toutes les images", set(M) == (ids["kb"] | ids["val"] | ids["test"]))
    check("attributs texte uniquement pour la KB", all((M[i]["caption_attributes"] is None) == (M[i]["split"] != "kb") for i in M))

    # ---- 4. embeddings + index
    import faiss
    for enc in cfg["encoders"]:
        d = p["emb"] / enc
        if not (d / "kb_img.npy").exists():
            continue
        for s in ("kb", "val", "test"):
            X, I_ = np.load(d / f"{s}_img.npy"), np.load(d / f"ids_{s}.npy")
            good = np.isfinite(X).all() and len(X) == len(I_) == sz[s] and set(I_.tolist()) == ids[s] \
                and np.allclose(np.linalg.norm(X, axis=1), 1, atol=1e-3) and len(set(I_.tolist())) == len(I_)
            check(f"embeddings {enc}/{s}", bool(good), f"shape={X.shape}")
        fl = faiss.read_index(str(p["index"] / f"{enc}_flatip.faiss")); Xkb = np.load(d / "kb_img.npy")
        _, Ik = fl.search(Xkb[:200], 1)
        check(f"index {enc} : ntotal=KB et auto-retrouvabilité", fl.ntotal == sz["kb"] and (Ik[:, 0] == np.arange(len(Ik))).mean() > 0.99)

    # ---- 5. audit de fuite par retrieval (encodeur principal)
    enc = cfg["primary_encoder"]
    near_dh = {(n["query_id"], n["match_id"]) for n in read_json(p["audit"] / "image_audit.json")["near_duplicates_eval_vs_others"] if n["match_split"] == "kb" or n["query_split"] == "kb"}
    for s in ("val", "test"):
        Q, qid = R.image_embeddings(s), R.ids(s)
        S, I = R.search_images(Q, k=10, query_ids=qid)      # lève AssertionError en cas d'auto-retrouvaille
        check(f"ANTI-FUITE : {s} jamais retrouvé pour lui-même (k=10)", True)
        sim = S[:, 0]; hi = int((sim >= 0.97).sum())
        dh = {q for q, m in near_dh if q in set(qid.tolist())}
        check(f"info : {s} avec near-duplicate dans la KB", True, f"cos>=0.97: {hi}/{len(qid)} | dHash<={cfg['image']['near_dup_hamming']}: {len(dh)}")
        if hi or dh:
            RESULTS[-1]["flagged_ids"] = sorted(set(qid[sim >= 0.97].tolist()) | dh)[:50]
        RESULTS[-1]["status"] = "WARN" if (hi or dh) else "PASS"

    # ---- 6. test de fumée : la release fonctionne avec le pipeline de retrieval (NN-caption baseline, CIDEr)
    rng = np.random.RandomState(0)
    for s in ("val", "test"):
        Q, qid = R.image_embeddings(s), R.ids(s)
        _, I = R.search_images(Q, k=5, query_ids=qid)
        refs = R.eval_refs(s)
        res = {int(i): R.kb_captions_of(I[n, 0])[0] for n, i in enumerate(qid)}
        rnd = {int(i): R.kb_captions_of(rng.choice(list(ids["kb"])))[0] for i in qid}
        c_nn, _ = cider_scores(res, refs); c_rd, _ = cider_scores(rnd, refs)
        jac = np.mean([len(set(M[int(q)]["category_counts"]) & set().union(*[set(M[int(j)]["category_counts"]) for j in I[n, :1]])) /
                       max(1, len(set(M[int(q)]["category_counts"]) | set().union(*[set(M[int(j)]["category_counts"]) for j in I[n, :1]]))) for n, q in enumerate(qid)])
        check(f"fumée {s} : NN-caption CIDEr > aléatoire", c_nn >= c_rd, f"NN={c_nn:.3f} random={c_rd:.3f} | Jaccard classes@1={jac:.3f}", blocking=False)
        RESULTS[-1]["metrics"] = {"cider_nn": c_nn, "cider_random": c_rd, "jaccard_cls_top1": float(jac)}

    n_fail = sum(r["status"] == "FAIL" for r in RESULTS); n_warn = sum(r["status"] == "WARN" for r in RESULTS)
    rep = {"version": cfg["release_version"], "passed": n_fail == 0, "n_fail": n_fail, "n_warn": n_warn, "checks": RESULTS}
    write_json(p["qc"] / "qc_report.json", rep)
    md = [f"# QC — {cfg['release_version']}", f"**{'PASS' if n_fail == 0 else 'FAIL'}** — {len(RESULTS)} contrôles, {n_fail} échec(s), {n_warn} avertissement(s)", "",
          "| contrôle | statut | détail |", "|---|---|---|"] + [f"| {r['check']} | {r['status']} | {r['detail']} |" for r in RESULTS]
    (p["qc"] / "qc_report.md").write_text("\n".join(md), encoding="utf-8")
    log.info("QC : %d échec(s), %d avertissement(s)", n_fail, n_warn)
    sys.exit(1 if n_fail else 0)

## Lancer l'étape

In [20]:
import sys
sys.argv = ["notebook"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

21:35:40 | INFO | taille split kb                                PASS 10000 vs 10000
21:35:40 | INFO | hash des IDs kb = splits_meta                  PASS 
21:35:40 | INFO | taille split val                               PASS 100 vs 100
21:35:40 | INFO | hash des IDs val = splits_meta                 PASS 
21:35:40 | INFO | taille split test                              PASS 100 vs 100
21:35:40 | INFO | hash des IDs test = splits_meta                PASS 
21:35:40 | INFO | splits deux à deux disjoints                   PASS 
21:35:40 | INFO | images invalides (corrompues/absentes)         PASS 0
21:35:40 | INFO | IDs uniques dans images.csv                    PASS 
21:35:40 | INFO | doublons exacts (sha256) inter/intra splits    PASS 0 images
21:35:40 | INFO | pas de NaN dans les stats pixel                PASS 
21:35:41 | INFO | KB : 5 captions / image                        PASS min=5 max=5
21:35:41 | INFO | val : 5 références / image                     PASS 
21:35:41 | INFO | test 

## Résultats

In [21]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Rapport QC

In [22]:
q = json.load(open(P["qc"] / "qc_report.json"))
print("PASS" if q["passed"] else "FAIL", "| échecs:", q["n_fail"], "| avertissements:", q["n_warn"])
t = pd.DataFrame([{k: c[k] for k in ("check", "status", "detail")} for c in q["checks"]])
pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 200); display(t)

PASS | échecs: 0 | avertissements: 0


,check,status,detail
0,taille split kb,PASS,10000 vs 10000
1,hash des IDs kb = splits_meta,PASS,
2,taille split val,PASS,100 vs 100
3,hash des IDs val = splits_meta,PASS,
4,taille split test,PASS,100 vs 100
5,hash des IDs test = splits_meta,PASS,
6,splits deux Ã deux disjoints,PASS,
7,images invalides (corrompues/absentes),PASS,0
8,IDs uniques dans images.csv,PASS,
9,doublons exacts (sha256) inter/intra splits,PASS,0 images
